# Trabalho 1: Panoramas

**MO446 Visão Computacional**, Instituto de Computação, Unicamp

**Alunos:** Beatriz Celante (RA 194568), Felipe (RA 236106), Gustavo Uchôa (RA 300637), Lucas Miranda (RA 322985)<br>
**Professor:** Prof. Dr. Anderson Rocha

Replicação do repositório [becelante/mo-446-visao-computacional](https://github.com/becelante/mo-446-visao-computacional) em um único notebook.

**Como executar:** coloque os `.ARW` do [Drive](https://drive.google.com/drive/folders/1qkxJ3tsieq7-eGgap9bC-92S4rU1gImP?usp=sharing) em `D1/`, `A1/` e `B1/` e execute as células em ordem. As fotos reveladas ficam em `D1_png/`, `A1_png/` e `B1_png/`; os resultados, em `D1_results/` (Seções 2 a 6 do relatório) e `extras/` (Extras 1 a 4).

## Configuração

In [ ]:
import csv
import ctypes
import gc
import math
import os
import random
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import numpy as np
import rawpy
import torch
from IPython.display import Image, Markdown, display
from lightglue import LightGlue, SuperPoint
from lightglue.utils import rbd
from scipy.optimize import least_squares
from scipy.sparse import coo_matrix

In [ ]:
ROOT = Path.cwd()
SEED = 0

RAW_SETS = [("D1", 1600, 8, True), ("A1", 0, 16, False), ("B1", 0, 16, False)]

DETECTOR = "SIFT"
DETECTORS_COMPARED = ("SIFT", "ORB")
RATIO_THRESH = 0.75
MIN_MATCHES = 8
BETA = 0.3

BASE_INPUT = ROOT / "D1_png"
BASE_RESULTS = ROOT / "D1_results"
BASE_RANSAC_THRESH = 1.0
BASE_MIN_INLIERS = 10
FEATHER_WIDTH = 15
MOSAICOS = (1, 4, 8)

EXTRAS_RESULTS = ROOT / "extras"
RANSAC_THRESH = 4.0
MIN_INLIERS = 20
WORK_DIM = 3000
COMPOSE_SCALE = 1.0
NUM_BANDS = 7
LOW_PARALLAX = {"ba": {"max_pair_error": 4.0, "loss_scale": 1.0},
                "align": {"fade": 48, "max_flow": 12.0, "smooth": 3.0}}
HIGH_PARALLAX = {"ba": {"max_pair_error": float("inf"), "loss_scale": 10.0},
                 "align": {"fade": 150, "max_flow": 80.0, "smooth": 12.0}}
EXTRAS = {"D1": ("cylindrical", LOW_PARALLAX),
          "A1": ("cylindrical", LOW_PARALLAX),
          "B1": ("spherical", HIGH_PARALLAX)}
X4_DATASET = "A1"

In [ ]:
def out(*parts):
    path = Path(*parts)
    path.parent.mkdir(parents=True, exist_ok=True)
    return str(path)


def write_csv(path, rows):
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def table(rows):
    keys = list(rows[0].keys())
    lines = ["| " + " | ".join(map(str, keys)) + " |", "|" + "---|" * len(keys)]
    lines += ["| " + " | ".join(str(row[k]) for k in keys) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


def show(img, width=1200):
    if img.shape[1] > width:
        img = cv2.resize(img, (width, round(img.shape[0] * width / img.shape[1])), interpolation=cv2.INTER_AREA)
    display(Image(data=cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 90])[1].tobytes()))


def free_memory():
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except OSError:
        pass

## 1. Coleta das imagens

Revelação dos RAW com balanço de branco (mediana do "as shot") e brilho únicos por conjunto. O D1 é reduzido a 1600 px e recebe nomes aleatórios; os originais ficam em `nomes_originais.csv`, usados só para rotular as saídas. A1 e B1 ficam em resolução total e 16 bits.

In [ ]:
SRGB_GAMMA = (2.4, 12.92)
NAMES_FILE = "nomes_originais.csv"
LIBRAW_LOCK = threading.Lock()  # o DHT do LibRaw em paralelo muda alguns pixels entre execucoes


def common_white_balance(paths):
    wbs = []
    for path in paths:
        with rawpy.imread(str(path)) as raw:
            wbs.append(raw.camera_whitebalance)
    wb = np.median(np.array(wbs, dtype=np.float64), axis=0)
    if wb[3] == 0:
        wb[3] = wb[1]
    return [float(v) for v in wb]


def bright_percentile(path, user_wb):
    with LIBRAW_LOCK, rawpy.imread(str(path)) as raw:
        rgb = raw.postprocess(user_wb=user_wb, no_auto_bright=True, gamma=(1, 1), output_bps=16, half_size=True)
    lum = rgb.astype(np.float32) @ np.float32([0.2126, 0.7152, 0.0722]) / 65535.0
    return float(np.percentile(lum, 99))


def develop(path, out_path, user_wb, bright, max_dim, bits):
    with LIBRAW_LOCK, rawpy.imread(str(path)) as raw:
        rgb = raw.postprocess(demosaic_algorithm=rawpy.DemosaicAlgorithm.DHT, user_wb=user_wb,
                              no_auto_bright=True, bright=bright, gamma=SRGB_GAMMA, output_bps=16)
    h, w = rgb.shape[:2]
    if max_dim and max(h, w) > max_dim:
        scale = max_dim / max(h, w)
        rgb = cv2.resize(rgb, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
    if bits == 8:
        rgb = np.round(rgb / 257.0).astype(np.uint8)
    cv2.imwrite(str(out_path), cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_PNG_COMPRESSION, 1])


def convert_arw_to_png(input_dir, output_dir, max_dim, bits, anonymize):
    arw_files = sorted(f for f in os.listdir(input_dir) if f.lower().endswith(".arw"))
    output_dir.mkdir(exist_ok=True)
    for old in output_dir.iterdir():
        if old.suffix.lower() == ".png" or old.name == NAMES_FILE:
            old.unlink()
    paths = [input_dir / f for f in arw_files]
    stems = [Path(f).stem for f in arw_files]
    if anonymize:
        rng = random.Random(0)
        stems = [f"{rng.getrandbits(32):08x}" for _ in arw_files]
        with open(output_dir / NAMES_FILE, "w", newline="") as f:
            csv.writer(f).writerows([("arquivo", "original")] + [(s + ".png", a) for s, a in zip(stems, arw_files)])
    n = len(paths)
    user_wb = common_white_balance(paths)
    with ThreadPoolExecutor(min(8, os.cpu_count())) as ex:
        p99 = list(ex.map(bright_percentile, paths, [user_wb] * n))
        bright = float(np.clip(1.0 / max(np.median(p99), 1e-6), 1.0, 64.0))
        list(ex.map(develop, paths, [output_dir / f"{s}.png" for s in stems], [user_wb] * n, [bright] * n,
                    [max_dim] * n, [bits] * n))
    print(f"{input_dir.name}: {n} fotos, balanco de branco {[round(v, 1) for v in user_wb]}, brilho {bright:.3f}")

In [ ]:
for name, max_dim, bits, anonymize in RAW_SETS:
    convert_arw_to_png(ROOT / name, ROOT / f"{name}_png", max_dim, bits, anonymize)

### Carregamento do D1

A ordem de leitura é embaralhada; a ordenação usa só o conteúdo das imagens.

In [ ]:
filenames = sorted(f for f in os.listdir(BASE_INPUT) if f.endswith(".png"))
random.Random(SEED).shuffle(filenames)
images = [cv2.imread(str(BASE_INPUT / f)) for f in filenames]
with open(BASE_INPUT / NAMES_FILE) as f:
    labels = {row["arquivo"]: Path(row["original"]).stem for row in csv.DictReader(f)}
names = [labels[f] for f in filenames]
print("Ordem de leitura:", names)

## 2. Detecção e extração de características

In [ ]:
def to_uint8(image):
    if image.dtype == np.uint8:
        return image
    return cv2.convertScaleAbs(image, alpha=255.0 / np.iinfo(image.dtype).max)


def to_gray8(image, equalize=False):
    gray = to_uint8(cv2.cvtColor(image, cv2.COLOR_BGR2GRAY))
    if equalize:
        gray = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    return gray


def detect_features(image, method="SIFT", equalize=False):
    detector = cv2.SIFT_create() if method == "SIFT" else cv2.ORB_create(nfeatures=4000)
    return detector.detectAndCompute(to_gray8(image, equalize), None)


def draw_keypoints(image, keypoints):
    return cv2.drawKeypoints(to_uint8(image), keypoints, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

In [ ]:
all_kps, all_descs = zip(*(detect_features(img, DETECTOR) for img in images))
table([{"imagem": n, "keypoints": len(k)} for n, k in zip(names, all_kps)])

## 3. Emparelhamento de características

FLANN (5 árvores, 50 checks) para o SIFT e força bruta com Hamming para o ORB; k = 2 e ratio test de Lowe com τ = 0,75.

In [ ]:
def knn_match(descs1, descs2, method="SIFT"):
    if descs1 is None or descs2 is None or len(descs1) < 2 or len(descs2) < 2:
        return []
    cv2.setRNGSeed(0)
    if method == "SIFT":
        matcher = cv2.FlannBasedMatcher(dict(algorithm=1, trees=5), dict(checks=50))
        descs1, descs2 = descs1.astype(np.float32), descs2.astype(np.float32)
    else:
        matcher = cv2.BFMatcher(cv2.NORM_HAMMING)
    return [p for p in matcher.knnMatch(descs1, descs2, k=2) if len(p) == 2]


def match_features(descs1, descs2, method="SIFT", ratio_thresh=RATIO_THRESH):
    good = [m for m, n in knn_match(descs1, descs2, method) if m.distance < ratio_thresh * n.distance]
    return sorted(good, key=lambda m: m.distance)


def draw_matches_before_after(img1, kps1, img2, kps2, all_matches, good, inlier_mask, max_draw=150, seed=0):
    rng = np.random.default_rng(seed)
    flags = cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    idx = rng.choice(len(all_matches), min(max_draw, len(all_matches)), replace=False)
    before = cv2.drawMatches(img1, kps1, img2, kps2, [all_matches[i] for i in idx], None,
                             matchColor=(0, 255, 255), flags=flags)
    idx = rng.choice(len(good), min(max_draw, len(good)), replace=False)
    after = cv2.drawMatches(img1, kps1, img2, kps2, [good[i] for i in idx if inlier_mask[i]], None,
                            matchColor=(0, 200, 0), flags=flags)
    after = cv2.drawMatches(img1, kps1, img2, kps2, [good[i] for i in idx if not inlier_mask[i]], after,
                            matchColor=(0, 0, 255), flags=flags | cv2.DrawMatchesFlags_DRAW_OVER_OUTIMG)
    return before, after

## 4. Ordenação automática das imagens

RANSAC em todos os pares, com a verificação de Brown e Lowe (inliers > 8 + 0,3 · matches); matriz de conectividade, grafo de vizinhança, rejeição das intrusas e ordem gulosa.

In [ ]:
NO_HOMOGRAPHY = {"n_inliers": 0, "inlier_ratio": 0.0, "mean_reproj_error": float("nan")}


def estimate_homography(kps1, kps2, matches, ransac_thresh):
    if len(matches) < 4:
        return None, None, NO_HOMOGRAPHY
    pts1 = np.float32([kps1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    pts2 = np.float32([kps2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
    H, mask = cv2.findHomography(pts1, pts2, cv2.RANSAC, ransac_thresh)
    if H is None:
        return None, None, NO_HOMOGRAPHY
    mask = mask.ravel().astype(bool)
    errors = np.linalg.norm(cv2.perspectiveTransform(pts1[mask], H) - pts2[mask], axis=2).ravel()
    return H, mask, {"n_inliers": int(mask.sum()), "inlier_ratio": float(mask.mean()),
                     "mean_reproj_error": float(errors.mean()) if errors.size else float("nan")}


def count_inliers(kps1, descs1, kps2, descs2, ransac_thresh, beta):
    good = match_features(descs1, descs2)
    if len(good) < MIN_MATCHES:
        return 0, good, None, None
    H, mask, metrics = estimate_homography(kps1, kps2, good, ransac_thresh)
    if H is None or metrics["n_inliers"] <= 8 + beta * len(good):
        return 0, good, None, None
    cx, cy = np.float32([kps1[m.queryIdx].pt for m, ok in zip(good, mask) if ok]).mean(axis=0)
    local_scale = np.linalg.det(H) / (H[2, 0] * cx + H[2, 1] * cy + H[2, 2]) ** 3
    if not 0.2 < local_scale < 5:
        return 0, good, None, None
    return metrics["n_inliers"], good, H, mask


def build_connectivity_matrix(all_kps, all_descs, ransac_thresh, beta):
    n = len(all_kps)
    pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
    connectivity = np.zeros((n, n), dtype=int)
    homographies, inlier_points = {}, {}
    with ThreadPoolExecutor(os.cpu_count()) as ex:
        results = ex.map(lambda p: count_inliers(all_kps[p[0]], all_descs[p[0]], all_kps[p[1]], all_descs[p[1]],
                                                 ransac_thresh, beta), pairs)
        for (i, j), (n_inliers, good, H, mask) in zip(pairs, results):
            connectivity[i, j] = connectivity[j, i] = n_inliers
            if H is not None:
                homographies[(i, j)] = H
                inliers = [m for m, ok in zip(good, mask) if ok]
                inlier_points[(i, j)] = (np.float32([all_kps[i][m.queryIdx].pt for m in inliers]),
                                         np.float32([all_kps[j][m.trainIdx].pt for m in inliers]))
    return connectivity, homographies, inlier_points


def build_neighbor_graph(connectivity, min_inliers):
    n = connectivity.shape[0]
    return {i: [(j, connectivity[i, j]) for j in range(n) if i != j and connectivity[i, j] >= min_inliers]
            for i in range(n)}


def detect_intruders(graph):
    return [node for node, neighbors in graph.items() if not neighbors]


def infer_order(graph, valid_nodes):
    sub_graph = {n: [(nb, w) for nb, w in graph[n] if nb in valid_nodes] for n in valid_nodes}
    current = min(valid_nodes, key=lambda n: len(sub_graph[n]))
    order, visited = [current], {current}
    while len(order) < len(valid_nodes):
        candidates = [(nb, w) for nb, w in sub_graph[current] if nb not in visited]
        if candidates:
            current = max(candidates, key=lambda x: x[1])[0]
        else:
            current = next(n for n in valid_nodes if n not in visited)
        order.append(current)
        visited.add(current)
    return order

In [ ]:
def draw_neighbor_graph(connectivity, order, intruders, names, min_inliers, size=1000):
    img = np.full((size, size, 3), 255, np.uint8)
    blue, gray, red, black = (180, 110, 30), (205, 205, 205), (80, 40, 210), (40, 40, 40)
    font, c, radius = cv2.FONT_HERSHEY_SIMPLEX, size / 2, 0.36 * size
    nodes = list(order) + list(intruders)
    pos = {}
    for k, node in enumerate(nodes):
        ang = 2 * math.pi * k / len(nodes) - math.pi / 2
        pos[node] = (int(c + radius * math.cos(ang)), int(c + radius * math.sin(ang)))
    for i in order:
        for j in order:
            if i < j and connectivity[i, j] >= min_inliers:
                cv2.line(img, pos[i], pos[j], gray, 1, cv2.LINE_AA)
    top = max(connectivity[a, b] for a, b in zip(order, order[1:]))
    for a, b in zip(order, order[1:]):
        w = int(connectivity[a, b])
        cv2.line(img, pos[a], pos[b], blue, 2 + int(6 * w / top), cv2.LINE_AA)
        mx, my = (pos[a][0] + pos[b][0]) // 2, (pos[a][1] + pos[b][1]) // 2
        dx, dy = mx - c, my - c
        norm = max(math.hypot(dx, dy), 1)
        cv2.putText(img, str(w), (int(mx + 30 * dx / norm) - 20, int(my + 30 * dy / norm) + 5),
                    font, 0.55, blue, 2, cv2.LINE_AA)
    for k, node in enumerate(nodes):
        x, y = pos[node]
        color = red if node in intruders else blue
        label = "X" if node in intruders else str(k + 1)
        cv2.circle(img, (x, y), 24, (255, 255, 255), -1, cv2.LINE_AA)
        cv2.circle(img, (x, y), 24, color, 3, cv2.LINE_AA)
        (tw, th), _ = cv2.getTextSize(label, font, 0.7, 2)
        cv2.putText(img, label, (x - tw // 2, y + th // 2), font, 0.7, color, 2, cv2.LINE_AA)
        dx, dy = x - c, y - c
        norm = max(math.hypot(dx, dy), 1)
        name = names[node] + (" (intrusa)" if node in intruders else "")
        (tw, _), _ = cv2.getTextSize(name, font, 0.5, 1)
        ux, uy = dx / norm, dy / norm
        d = 32 + tw / 2 * abs(ux) + 8 * abs(uy)
        cv2.putText(img, name, (int(x + d * ux - tw / 2), int(y + d * uy + 5)), font, 0.5,
                    red if node in intruders else black, 1, cv2.LINE_AA)
    return img

In [ ]:
connectivity, homographies, _ = build_connectivity_matrix(all_kps, all_descs, BASE_RANSAC_THRESH, BETA)
matrix = [{"": n, **{m: int(v) for m, v in zip(names, row)}} for n, row in zip(names, connectivity)]
write_csv(out(BASE_RESULTS, "4_ordenacao", "matriz_conectividade.csv"), matrix)
table(matrix)

In [ ]:
graph = build_neighbor_graph(connectivity, BASE_MIN_INLIERS)
intruders = detect_intruders(graph)
order = infer_order(graph, [i for i in range(len(images)) if i not in intruders])
print("Intrusas:", [names[i] for i in intruders])
print("Ordem inferida:", [names[i] for i in order])

graph_img = draw_neighbor_graph(connectivity, order, intruders, names, BASE_MIN_INLIERS)
cv2.imwrite(out(BASE_RESULTS, "4_ordenacao", "grafo_vizinhanca.png"), graph_img)
for i in intruders:
    cv2.imwrite(out(BASE_RESULTS, "4_ordenacao", f"intrusa_{names[i]}.jpg"), images[i])
show(np.hstack([cv2.resize(images[i], (1500, 1000)) for i in intruders] + [cv2.resize(graph_img, (1000, 1000))]), 900)

### Par central da sequência

Comparação SIFT x ORB (Seção 2) e matches antes e depois do ratio test (Seção 3).

In [ ]:
a, b = order[len(order) // 2 - 1], order[len(order) // 2]
rows, keypoint_imgs = [], []
for method in DETECTORS_COMPARED:
    kps_a, descs_a = detect_features(images[a], method)
    kps_b, descs_b = detect_features(images[b], method)
    good = match_features(descs_a, descs_b, method)
    _, _, metrics = estimate_homography(kps_a, kps_b, good, BASE_RANSAC_THRESH)
    rows.append({"detector": method, "keypoints": len(kps_a),
                 "matches_brutos": len(knn_match(descs_a, descs_b, method)), "matches_ratio_test": len(good),
                 "inliers": metrics["n_inliers"], "taxa_inliers": round(metrics["inlier_ratio"], 3),
                 "erro_reproj_px": round(metrics["mean_reproj_error"], 3)})
    keypoint_imgs.append(draw_keypoints(images[a], kps_a))
    cv2.imwrite(out(BASE_RESULTS, "2_deteccao", f"{names[a]}_{method}.jpg"), keypoint_imgs[-1])
write_csv(out(BASE_RESULTS, "2_deteccao", "comparacao_detectores.csv"), rows)
print(f"Par central: {names[a]} e {names[b]}")
table(rows)
show(np.hstack(keypoint_imgs))

In [ ]:
all_matches = [m for m, _ in knn_match(all_descs[a], all_descs[b])]
good = match_features(all_descs[a], all_descs[b])
_, mask, _ = estimate_homography(all_kps[a], all_kps[b], good, BASE_RANSAC_THRESH)
before, after = draw_matches_before_after(images[a], all_kps[a], images[b], all_kps[b], all_matches, good, mask)
cv2.imwrite(out(BASE_RESULTS, "3_emparelhamento", "matches_antes.jpg"), before)
cv2.imwrite(out(BASE_RESULTS, "3_emparelhamento", "matches_depois.jpg"), after)
print(f"{len(all_matches)} matches brutos, {len(good)} apos o ratio test")
show(before)
show(after)

## 5. Estimação de homografia e alinhamento

Métricas dos pares consecutivos, homografias encadeadas até a imagem central e warp para o canvas comum.

In [ ]:
pair_rows = []
for i, j in zip(order, order[1:]):
    good = match_features(all_descs[i], all_descs[j])
    _, _, metrics = estimate_homography(all_kps[i], all_kps[j], good, BASE_RANSAC_THRESH)
    pair_rows.append({"imagem_1": names[i], "imagem_2": names[j],
                      "matches_brutos": len(knn_match(all_descs[i], all_descs[j])), "matches_ratio_test": len(good),
                      "inliers": metrics["n_inliers"], "taxa_inliers": round(metrics["inlier_ratio"], 4),
                      "erro_reproj_medio_px": round(metrics["mean_reproj_error"], 4)})
write_csv(out(BASE_RESULTS, "5_alinhamento", "metricas.csv"), pair_rows)
table(pair_rows)

In [ ]:
def get_H(a, b, homographies):
    if (a, b) in homographies:
        return homographies[(a, b)]
    if (b, a) in homographies:
        return np.linalg.inv(homographies[(b, a)])
    return None


def compose_pairwise_homographies(order, homographies):
    ref_pos = len(order) // 2
    H_to_ref = {order[ref_pos]: np.eye(3)}
    for step, stop in ((1, len(order) - 1), (-1, 0)):
        for k in range(ref_pos, stop, step):
            i, j = order[k], order[k + step]
            if i not in H_to_ref:
                break
            H_i_to_j = get_H(i, j, homographies)
            if H_i_to_j is not None:
                H_to_ref[j] = H_to_ref[i] @ np.linalg.inv(H_i_to_j)
    return H_to_ref


def get_canvas_bounds(images, H_list):
    corners = []
    for img, H in zip(images, H_list):
        h, w = img.shape[:2]
        corners.append(cv2.perspectiveTransform(np.float32([[0, 0], [w, 0], [w, h], [0, h]]).reshape(-1, 1, 2), H))
    corners = np.concatenate(corners, axis=0)
    x_min, y_min = np.floor(corners.min(axis=0).ravel()).astype(int)
    x_max, y_max = np.ceil(corners.max(axis=0).ravel()).astype(int)
    translation = np.array([[1, 0, -x_min], [0, 1, -y_min], [0, 0, 1]], dtype=np.float64)
    return (int(x_max - x_min), int(y_max - y_min)), translation


def warp_to_canvas(img, H, translation, canvas_size):
    H_total = translation @ H
    warped = cv2.warpPerspective(img, H_total, canvas_size)
    mask = cv2.warpPerspective(np.full(img.shape[:2], 255, dtype=np.uint8), H_total, canvas_size)
    return warped, cv2.erode(mask, np.ones((5, 5), np.uint8))

In [ ]:
H_to_ref = compose_pairwise_homographies(order, homographies)
kept = [i for i in order if i in H_to_ref]
canvas_size, translation = get_canvas_bounds([images[i] for i in kept], [H_to_ref[i] for i in kept])
warped, masks = zip(*(warp_to_canvas(images[i], H_to_ref[i], translation, canvas_size) for i in kept))
print(f"Referencia: {names[order[len(order) // 2]]}; canvas {canvas_size[0]}x{canvas_size[1]} px")

## 6. Composição, blending e remoção de fantasmas

Composição ingênua (média) e incremental, com costura ótima por programação dinâmica e feathering de 15 px. Os mosaicos parciais da Seção 5 são salvos durante a composição incremental.

In [ ]:
def naive_composite(warped_images, masks):
    h, w = masks[0].shape
    acc = np.zeros((h, w, 3), dtype=np.float64)
    count = np.zeros((h, w, 1), dtype=np.float64)
    for img, mask in zip(warped_images, masks):
        m = (mask > 0).astype(np.float64)[..., None]
        acc += img.astype(np.float64) * m
        count += m
    count[count == 0] = 1
    return (acc / count).astype(np.uint8)


def find_optimal_seam(cost_map, valid_mask):
    h, w = cost_map.shape
    INF = 1e9
    cost = np.where(valid_mask, cost_map, INF).astype(np.float64)
    dp = cost.copy()
    backtrack = np.zeros((h, w), dtype=np.int8)
    offsets = np.array([0, -1, 1])
    for y in range(1, h):
        up = dp[y - 1]
        left = np.roll(up, 1)
        left[0] = INF
        right = np.roll(up, -1)
        right[-1] = INF
        stacked = np.stack([up, left, right], axis=0)
        best = np.argmin(stacked, axis=0)
        dp[y] = cost[y] + stacked[best, np.arange(w)]
        backtrack[y] = offsets[best]
    seam = np.zeros(h, dtype=int)
    seam[h - 1] = int(np.argmin(dp[h - 1]))
    for y in range(h - 2, -1, -1):
        seam[y] = int(np.clip(seam[y + 1] + backtrack[y + 1, seam[y + 1]], 0, w - 1))
    return seam


def blend_pair_with_seam(base, base_mask, new_img, new_mask, feather_width):
    overlap = (base_mask > 0) & (new_mask > 0)
    only_new = (new_mask > 0) & ~overlap
    result = base.copy()
    result[only_new] = new_img[only_new]
    if not overlap.any():
        return result, base_mask | new_mask

    ys, xs = np.where(overlap)
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    base_roi = base[y0:y1, x0:x1].astype(np.float64)
    new_roi = new_img[y0:y1, x0:x1].astype(np.float64)
    overlap_roi = overlap[y0:y1, x0:x1]

    seam_x = find_optimal_seam(np.sum((base_roi - new_roi) ** 2, axis=2), overlap_roi)
    dist = np.arange(overlap_roi.shape[1])[None, :] - seam_x[:, None]
    weight = np.clip((dist + feather_width / 2) / feather_width, 0, 1)[..., None]
    blended = base_roi * (1 - weight) + new_roi * weight
    result[y0:y1, x0:x1][overlap_roi] = blended[overlap_roi].astype(np.uint8)
    return result, base_mask | new_mask


def ghost_comparison(naive, panorama, size=(700, 450)):
    w, h = size
    valid = (naive.max(axis=2) > 0) & (panorama.max(axis=2) > 0)
    diff = np.abs(naive.astype(np.float32) - panorama.astype(np.float32)).mean(axis=2) * valid
    score = cv2.boxFilter(diff, -1, (w, h))
    score[cv2.erode(valid.astype(np.uint8), np.ones((h, w), np.uint8)) == 0] = -1
    cy, cx = np.unravel_index(np.argmax(score), score.shape)
    x0, y0 = cx - w // 2, cy - h // 2
    gap = np.full((h, 12, 3), 255, np.uint8)
    return np.hstack([naive[y0:y0 + h, x0:x0 + w], gap, panorama[y0:y0 + h, x0:x0 + w]])


def overlap_metrics(warped, masks, names, thresh=30):
    rows = []
    for k in range(len(warped) - 1):
        a, b = warped[k], warped[k + 1]
        overlap = cv2.erode(((masks[k] > 0) & (masks[k + 1] > 0)).astype(np.uint8), np.ones((5, 5), np.uint8)) > 0
        d = np.abs(a.astype(np.float32) - b.astype(np.float32)).mean(axis=2)[overlap]
        la = cv2.cvtColor(a, cv2.COLOR_BGR2GRAY)[overlap].astype(np.float32).mean()
        lb = cv2.cvtColor(b, cv2.COLOR_BGR2GRAY)[overlap].astype(np.float32).mean()
        rows.append({"imagem_1": names[k], "imagem_2": names[k + 1],
                     "diferenca_media": round(float(d.mean()), 2),
                     "fracao_inconsistente": round(float((d > thresh).mean()), 4),
                     "razao_brilho": round(float(max(la, lb) / min(la, lb)), 4)})
    return rows

In [ ]:
naive = naive_composite(warped, masks)
panorama, panorama_mask = warped[0].copy(), masks[0].copy()
mosaics = []
for n in range(1, len(warped) + 1):
    if n > 1:
        panorama, panorama_mask = blend_pair_with_seam(panorama, panorama_mask, warped[n - 1], masks[n - 1],
                                                       FEATHER_WIDTH)
    if n in MOSAICOS:
        f = 2400 / panorama.shape[1]
        mosaics.append(cv2.resize(panorama, None, fx=f, fy=f, interpolation=cv2.INTER_AREA))
        cv2.imwrite(out(BASE_RESULTS, "5_alinhamento", f"mosaico_{n:02d}.jpg"), mosaics[-1])
show(np.vstack(mosaics), 900)

In [ ]:
ghosts = ghost_comparison(naive, panorama)
cv2.imwrite(out(BASE_RESULTS, "6_composicao", "comparacao_fantasmas.jpg"), ghosts)
show(ghosts)

overlap_rows = overlap_metrics(warped, masks, [names[i] for i in kept])
write_csv(out(BASE_RESULTS, "6_composicao", "metricas.csv"), overlap_rows)
table(overlap_rows)

In [ ]:
ys, xs = np.nonzero(panorama.max(axis=2) > 0)
panorama_final = panorama[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
cv2.imwrite(out(BASE_RESULTS, "6_composicao", "panorama_final.jpg"), panorama_final, [cv2.IMWRITE_JPEG_QUALITY, 92])
show(panorama_final)

## 7. Extras

Bundle adjustment (Extra 1), projeção cilíndrica ou esférica com volta de 360° (Extra 2), compensação de ganho e vinheta com blending multibanda (Extra 3) e SuperPoint + LightGlue (Extra 4). As etapas 2 a 5 rodam numa cópia com 3000 px; só a composição lê a resolução total.

### Ordenação espectral

Para muitas imagens: ordem pelo vetor de Fiedler do Laplaciano do grafo, ou pelo ângulo nos dois primeiros autovetores quando o grafo fecha um anel (360°), corrigida com trocas entre vizinhos.

In [ ]:
def connected_components(sub_graph, nodes):
    components, seen = [], set()
    for start in nodes:
        if start in seen:
            continue
        stack, comp = [start], []
        seen.add(start)
        while stack:
            node = stack.pop()
            comp.append(node)
            for nb, _ in sub_graph[node]:
                if nb not in seen:
                    seen.add(nb)
                    stack.append(nb)
        components.append(sorted(comp))
    return sorted(components, key=len, reverse=True)


def refine_path(seq, W, ring):
    seq, n = list(seq), len(seq)

    def w(a, b):
        return W[seq[a % n], seq[b % n]] if ring or (0 <= a < n and 0 <= b < n) else 0

    improved = True
    while improved:
        improved = False
        for k in range(n if ring else n - 1):
            if w(k - 1, k + 1) + w(k, k + 2) > w(k - 1, k) + w(k + 1, k + 2):
                seq[k % n], seq[(k + 1) % n] = seq[(k + 1) % n], seq[k % n]
                improved = True
    return seq


def infer_order_spectral(graph, valid_nodes):
    valid = set(valid_nodes)
    sub_graph = {n: [(nb, w) for nb, w in graph[n] if nb in valid] for n in valid_nodes}
    order, closed = [], False
    for c, comp in enumerate(connected_components(sub_graph, valid_nodes)):
        if len(comp) <= 2:
            order.extend(comp)
            continue
        index = {node: k for k, node in enumerate(comp)}
        W = np.zeros((len(comp), len(comp)))
        for node in comp:
            for nb, w in sub_graph[node]:
                W[index[node], index[nb]] = w
        _, eigvecs = np.linalg.eigh(np.diag(W.sum(axis=1)) - W)
        seq = list(np.argsort(np.arctan2(eigvecs[:, 2], eigvecs[:, 1])))
        links = [W[seq[k], seq[(k + 1) % len(seq)]] for k in range(len(seq))]
        is_ring = len(comp) >= 4 and min(links) > 0
        if is_ring:
            seq = refine_path(seq, W, ring=True)
            links = [W[seq[k], seq[(k + 1) % len(seq)]] for k in range(len(seq))]
            cut = int(np.argmin(links))
            seq = seq[cut + 1:] + seq[:cut + 1]
        else:
            seq = refine_path(list(np.argsort(eigvecs[:, 1])), W, ring=False)
            if len(sub_graph[comp[seq[-1]]]) < len(sub_graph[comp[seq[0]]]):
                seq.reverse()
        order.extend(comp[k] for k in seq)
        if c == 0:
            closed = is_ring
    return order, closed

### Modelo de câmera e chute inicial

Cada foto é uma câmera que só gira, com focal, ponto principal e distorção radial comuns. Focal e rotações iniciais saem das homografias.

In [ ]:
def make_camera(width, height, focal):
    return {"f": float(focal), "k1": 0.0, "k2": 0.0, "k3": 0.0, "width": int(width), "height": int(height),
            "cx": (width - 1) / 2.0, "cy": (height - 1) / 2.0, "D": 0.5 * float(np.hypot(width, height))}


def points_to_full_res(pts, scale):
    return (np.asarray(pts, dtype=np.float64) + 0.5) / scale - 0.5


def homography_to_centered_full_res(H, scale, camera):
    A = np.array([[scale, 0, 0.5 * scale - 0.5], [0, scale, 0.5 * scale - 0.5], [0, 0, 1]])
    T = np.array([[1, 0, camera["cx"]], [0, 1, camera["cy"]], [0, 0, 1]])
    M = np.linalg.inv(A @ T) @ H @ (A @ T)
    return M / M[2, 2]


def radial_factor(r2, k1, k2, k3):
    return 1 + r2 * (k1 + r2 * (k2 + r2 * k3))


def undistort_points(pts, camera):
    u = np.asarray(pts, dtype=np.float64) - (camera["cx"], camera["cy"])
    r2 = (u ** 2).sum(axis=1) / camera["D"] ** 2
    return u * radial_factor(r2, camera["k1"], camera["k2"], camera["k3"])[:, None]


def distort_points(u, camera):
    D = camera["D"]
    r_d = np.linspace(0, 1.6 * D, 16384)
    r_u = r_d * radial_factor((r_d / D) ** 2, camera["k1"], camera["k2"], camera["k3"])
    monotonic = np.concatenate([[True], np.cumprod(np.diff(r_u) > 0).astype(bool)])
    r_d, r_u = r_d[monotonic], r_u[monotonic]
    u = np.asarray(u, dtype=np.float64)
    r = np.sqrt((u ** 2).sum(axis=1))
    ratio = np.interp(r, r_u, r_d, right=np.nan) / np.maximum(r, 1e-12)
    ratio[r < 1e-9] = 1.0
    return u * ratio[:, None] + (camera["cx"], camera["cy"])


def rays(u, focal):
    v = np.concatenate([u / focal, np.ones((len(u), 1))], axis=1)
    return v / np.linalg.norm(v, axis=1, keepdims=True)


def rodrigues(rvecs):
    rvecs = np.atleast_2d(rvecs)
    theta = np.linalg.norm(rvecs, axis=1)
    k = rvecs / np.maximum(theta, 1e-12)[:, None]
    K = np.zeros((len(rvecs), 3, 3))
    K[:, 0, 1], K[:, 0, 2] = -k[:, 2], k[:, 1]
    K[:, 1, 0], K[:, 1, 2] = k[:, 2], -k[:, 0]
    K[:, 2, 0], K[:, 2, 1] = -k[:, 1], k[:, 0]
    s, c = np.sin(theta)[:, None, None], np.cos(theta)[:, None, None]
    return np.eye(3) + s * K + (1 - c) * (K @ K)


def nearest_rotation(M):
    U, _, Vt = np.linalg.svd(M)
    R = U @ Vt
    return -R if np.linalg.det(R) < 0 else R


def estimate_focal(homographies, scale, camera):
    candidates = []
    for H in homographies.values():
        h = homography_to_centered_full_res(H, scale, camera).ravel()
        estimates = []
        for d1, d2, v1n, v2n in (
            (h[6] * h[7], (h[7] - h[6]) * (h[7] + h[6]),
             -(h[0] * h[1] + h[3] * h[4]), h[0] ** 2 + h[3] ** 2 - h[1] ** 2 - h[4] ** 2),
            (h[0] * h[3] + h[1] * h[4], h[0] ** 2 + h[1] ** 2 - h[3] ** 2 - h[4] ** 2,
             -h[2] * h[5], h[5] ** 2 - h[2] ** 2),
        ):
            v1 = v1n / d1 if d1 != 0 else -1.0
            v2 = v2n / d2 if d2 != 0 else -1.0
            v1, v2 = max(v1, v2), min(v1, v2)
            if v1 > 0 and v2 > 0:
                estimates.append(np.sqrt(v1 if abs(d1) > abs(d2) else v2))
            elif v1 > 0:
                estimates.append(np.sqrt(v1))
        if len(estimates) == 2:
            candidates.append(np.sqrt(estimates[0] * estimates[1]))
    return float(np.median(candidates)) if candidates else None


def initial_rotations(homographies, connectivity, nodes, ref, scale, camera):
    K = np.diag([camera["f"], camera["f"], 1.0])
    K_inv = np.linalg.inv(K)
    rotations = {ref: np.eye(3)}
    nodes = set(nodes)
    while True:
        best = None
        for i in rotations:
            for j in nodes - rotations.keys():
                if connectivity[i, j] > 0 and (best is None or connectivity[i, j] > best[2]):
                    best = (i, j, connectivity[i, j])
        if best is None:
            return rotations
        i, j, _ = best
        R_ij = nearest_rotation(K_inv @ homography_to_centered_full_res(get_H(i, j, homographies), scale, camera) @ K)
        rotations[j] = R_ij @ rotations[i]

### Bundle adjustment

Mínimos quadrados robustos sobre os inliers de todos os pares; a segunda passada descarta pares inconsistentes. A correção de onda deixa o horizonte reto.

In [ ]:
def bundle_adjust(inlier_points, rotations, camera, scale, ref, max_points_per_pair=100, max_pair_error=4.0,
                  loss_scale=1.0):
    rng = np.random.default_rng(0)
    obs = []
    for (i, j), (pts_i, pts_j) in inlier_points.items():
        idx = np.arange(len(pts_i))
        if len(idx) > max_points_per_pair:
            idx = rng.choice(idx, max_points_per_pair, replace=False)
        obs.append((i, j, points_to_full_res(pts_i[idx], scale), points_to_full_res(pts_j[idx], scale)))

    camera = dict(camera)
    nodes = [ref] + sorted(n for n in rotations if n != ref)
    pos = {n: k for k, n in enumerate(nodes)}
    n_rot = 3 * (len(nodes) - 1)
    D = camera["D"]

    def ideal(P, k, c):
        u = P - c
        return u * radial_factor((u ** 2).sum(axis=1) / D ** 2, *k)[:, None]

    for pass_ in range(2):
        I = np.concatenate([np.full(len(p), pos[i]) for i, _, p, _ in obs])
        J = np.concatenate([np.full(len(p), pos[j]) for _, j, p, _ in obs])
        P_i = np.concatenate([p for _, _, p, _ in obs])
        P_j = np.concatenate([q for _, _, _, q in obs])
        R0 = np.stack([rotations[n] for n in nodes])
        f0, c0 = camera["f"], np.array([camera["cx"], camera["cy"]])

        def unpack(x):
            deltas = np.vstack([np.zeros(3), x[:n_rot].reshape(-1, 3)])
            return rodrigues(deltas) @ R0, f0 * x[n_rot], x[n_rot + 1:n_rot + 4], c0 + 100.0 * x[n_rot + 4:]

        def residuals(x):
            R, f, k, c = unpack(x)
            w_i = np.einsum("nlk,nl->nk", R[I], rays(ideal(P_i, k, c), f))
            w_j = np.einsum("nlk,nl->nk", R[J], rays(ideal(P_j, k, c), f))
            return (f * (w_i - w_j)).ravel()

        n_obs = len(I)
        base = 3 * np.arange(n_obs)
        rows, cols = [], []
        for cam in (I, J):
            sel = cam > 0
            for p in range(3):
                for q in range(3):
                    rows.append(base[sel] + p)
                    cols.append(3 * (cam[sel] - 1) + q)
        for p in range(3):
            for q in range(6):
                rows.append(base + p)
                cols.append(np.full(n_obs, n_rot + q))
        rows, cols = np.concatenate(rows), np.concatenate(cols)
        sparsity = coo_matrix((np.ones(len(rows)), (rows, cols)), shape=(3 * n_obs, n_rot + 6))

        x0 = np.concatenate([np.zeros(n_rot), [1.0, camera["k1"], camera["k2"], camera["k3"], 0.0, 0.0]])
        result = least_squares(residuals, x0, jac_sparsity=sparsity, method="trf", loss="soft_l1",
                               f_scale=loss_scale, x_scale="jac", max_nfev=100)
        R, f, k, c = unpack(result.x)
        rotations = {n: R[pos[n]] for n in nodes}
        camera.update(f=float(f), k1=float(k[0]), k2=float(k[1]), k3=float(k[2]), cx=float(c[0]), cy=float(c[1]))

        errors = np.linalg.norm(residuals(result.x).reshape(-1, 3), axis=1)
        stats, per_pair, offset = {}, [], 0
        for i, j, p, q in obs:
            e = errors[offset:offset + len(p)]
            offset += len(p)
            stats[(i, j)] = float(np.median(e))
            per_pair.append((i, j, p, q, e))
        print(f"  BA passada {pass_ + 1}: {len(obs)} pares, erro mediano {np.median(errors):.3f} px, f = {f:.1f} px")
        if pass_ == 0:
            keep = {pair for pair, med in stats.items() if med <= max_pair_error}
            for node in nodes:
                own = sorted((med, pair) for pair, med in stats.items() if node in pair)
                if sum(pair in keep for _, pair in own) < 2:
                    keep.update(pair for _, pair in own[:2])
            obs = []
            for i, j, p, q, e in per_pair:
                good = e <= max(3.0 * np.median(e), 2.0)
                if (i, j) in keep and good.sum() >= 8:
                    obs.append((i, j, p[good], q[good]))
    return rotations, camera, stats


def wave_correct(rotations):
    x_axes = np.array([R[0] for R in rotations.values()])
    _, eigvecs = np.linalg.eigh(x_axes.T @ x_axes)
    down = eigvecs[:, 0]
    if np.dot(down, np.array([R[1] for R in rotations.values()]).sum(axis=0)) < 0:
        down = -down
    forward = np.array([R[2] for R in rotations.values()]).sum(axis=0)
    forward -= np.dot(forward, down) * down
    forward /= np.linalg.norm(forward)
    W = np.stack([np.cross(down, forward), down, forward])
    return {k: R @ W.T for k, R in rotations.items()}

### Projeção cilíndrica ou esférica

In [ ]:
def srgb_to_linear(x):
    return np.where(x <= 0.04045, x / 12.92, ((x + 0.055) / 1.055) ** 2.4).astype(np.float32)


def linear_to_srgb(x):
    x = np.maximum(x, 0)
    return np.where(x <= 0.0031308, 12.92 * x, 1.055 * np.power(x, 1 / 2.4) - 0.055).astype(np.float32)


def to_proj(d, projection):
    x, y, z = d[..., 0], d[..., 1], d[..., 2]
    rho = np.hypot(x, z)
    return np.arctan2(x, z), (y / rho if projection == "cylindrical" else np.arctan2(y, rho))


def from_proj(a, b, projection):
    if projection == "cylindrical":
        return np.stack([np.sin(a), b, np.cos(a)], axis=-1)
    cb = np.cos(b)
    return np.stack([np.sin(a) * cb, np.sin(b), np.cos(a) * cb], axis=-1)


def wrap(a):
    return (a + np.pi) % (2 * np.pi) - np.pi


def make_canvas(rotations, camera, projection):
    w, h = camera["width"], camera["height"]
    t = np.linspace(0, 1, 129)
    border = np.concatenate([
        np.stack([t * (w - 1), np.zeros_like(t)], axis=1),
        np.stack([t * (w - 1), np.full_like(t, h - 1)], axis=1),
        np.stack([np.zeros_like(t), t * (h - 1)], axis=1),
        np.stack([np.full_like(t, w - 1), t * (h - 1)], axis=1),
    ])
    rays_cam = rays(undistort_points(border, camera), camera["f"])
    keys = sorted(rotations, key=lambda k: float(to_proj(rotations[k][2], projection)[0]))
    angles = np.array([float(to_proj(rotations[k][2], projection)[0]) for k in keys])
    gaps = np.diff(np.append(angles, angles[0] + 2 * np.pi))
    start = (int(np.argmax(gaps)) + 1) % len(keys)
    keys = keys[start:] + keys[:start]

    footprints, centers, prev = {}, {}, None
    for k in keys:
        a, b = to_proj(rays_cam @ rotations[k], projection)
        center = float(to_proj(rotations[k][2], projection)[0])
        if prev is not None:
            center = prev + wrap(center - prev)
        a = center + wrap(a - center)
        footprints[k], centers[k], prev = (a.min(), a.max(), b.min(), b.max()), center, center

    hfov = np.median([fp[1] - fp[0] for fp in footprints.values()])
    dup = None
    if len(keys) > 2 and gaps.max() < 0.5 * hfov:
        dup = max(keys) + 1
        a0, a1, b0, b1 = footprints[keys[0]]
        footprints[dup] = (a0 + 2 * np.pi, a1 + 2 * np.pi, b0, b1)
        centers[dup] = centers[keys[0]] + 2 * np.pi
        keys.append(dup)
    fp = np.array(list(footprints.values()))
    return {"f": camera["f"], "projection": projection, "order": keys, "dup": dup,
            "a_min": fp[:, 0].min(), "a_max": fp[:, 1].max(), "b_min": fp[:, 2].min(), "b_max": fp[:, 3].max(),
            "footprints": footprints, "centers": centers}


def canvas_size_at(canvas, s):
    fs = canvas["f"] * s
    return (int(np.ceil((canvas["a_max"] - canvas["a_min"]) * fs)),
            int(np.ceil((canvas["b_max"] - canvas["b_min"]) * fs)))


def image_roi(canvas, k, s):
    a0, a1, b0, b1 = canvas["footprints"][k]
    fs = canvas["f"] * s
    W, H = canvas_size_at(canvas, s)
    return (max(int(np.floor((a0 - canvas["a_min"]) * fs)), 0), max(int(np.floor((b0 - canvas["b_min"]) * fs)), 0),
            min(int(np.ceil((a1 - canvas["a_min"]) * fs)) + 1, W), min(int(np.ceil((b1 - canvas["b_min"]) * fs)) + 1, H))


def sample_offsets(offsets, s, x0, y0, w, rows):
    ox, oy, dx, dy, s_off = offsets
    r = s_off / s
    mx = ((x0 + np.arange(w) + 0.5) * r - 0.5 - ox).astype(np.float32)
    my = ((y0 + rows + 0.5) * r - 0.5 - oy).astype(np.float32)
    MX, MY = np.meshgrid(mx, my)
    sx = cv2.remap(dx, MX, MY, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    sy = cv2.remap(dy, MX, MY, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    return sx / r, sy / r


def warp_maps(R, camera, canvas, s, roi, src_scale, offsets=None, border=8, chunk_rows=256):
    x0, y0, x1, y1 = roi
    w, h = x1 - x0, y1 - y0
    fs = canvas["f"] * s
    xs = x0 + np.arange(w) + 0.5
    map_x = np.empty((h, w), np.float32)
    map_y = np.empty((h, w), np.float32)
    valid = np.empty((h, w), bool)
    r2 = np.empty((h, w), np.float32)
    c = np.array([camera["cx"], camera["cy"]])
    for r0 in range(0, h, chunk_rows):
        r1 = min(r0 + chunk_rows, h)
        X, Y = np.meshgrid(xs, y0 + np.arange(r0, r1) + 0.5)
        if offsets is not None:
            dx, dy = sample_offsets(offsets, s, x0, y0, w, np.arange(r0, r1))
            X, Y = X + dx, Y + dy
        cam = from_proj(canvas["a_min"] + X / fs, canvas["b_min"] + Y / fs, canvas["projection"]) @ R.T
        z = cam[..., 2]
        with np.errstate(divide="ignore", invalid="ignore"):
            u = camera["f"] * cam[..., :2] / z[..., None]
        p = distort_points(u.reshape(-1, 2), camera).reshape(r1 - r0, w, 2)
        ok = ((z > 1e-6) & np.isfinite(p).all(axis=-1)
              & (p[..., 0] >= border) & (p[..., 0] <= camera["width"] - 1 - border)
              & (p[..., 1] >= border) & (p[..., 1] <= camera["height"] - 1 - border))
        p = np.where(ok[..., None], p, -1.0)
        map_x[r0:r1] = (p[..., 0] + 0.5) * src_scale - 0.5
        map_y[r0:r1] = (p[..., 1] + 0.5) * src_scale - 0.5
        valid[r0:r1] = ok
        r2[r0:r1] = ((p - c) ** 2).sum(axis=-1) / camera["D"] ** 2
    return map_x, map_y, valid, r2


def vignetting_factor(r2, coeffs):
    return np.exp(r2 * (coeffs[0] + r2 * (coeffs[1] + r2 * coeffs[2])))


def warp_corrected(src, R, camera, canvas, s, roi, src_scale, gain=1.0, vignetting=(0, 0, 0),
                   interpolation=cv2.INTER_LINEAR, offsets=None, chunk_rows=512):
    x0, y0, x1, y1 = roi
    img = np.empty((y1 - y0, x1 - x0, 3), np.float32)
    valid = np.empty((y1 - y0, x1 - x0), bool)
    scale = np.float32(np.iinfo(src.dtype).max)
    for r0 in range(0, y1 - y0, chunk_rows):
        r1 = min(r0 + chunk_rows, y1 - y0)
        map_x, map_y, valid[r0:r1], r2 = warp_maps(R, camera, canvas, s, (x0, y0 + r0, x1, y0 + r1), src_scale, offsets)
        part = cv2.remap(src, map_x, map_y, interpolation, borderMode=cv2.BORDER_REPLICATE).astype(np.float32) / scale
        factor = 1.0 / (gain * vignetting_factor(r2, vignetting))
        img[r0:r1] = linear_to_srgb(srgb_to_linear(part) * factor[..., None])
    return img, valid


def resize_to_scale(img, full_size, s):
    w, h = full_size
    size = (max(int(round(w * s)), 1), max(int(round(h * s)), 1))
    if (img.shape[1], img.shape[0]) == size:
        return img
    return cv2.resize(img, size, interpolation=cv2.INTER_AREA)

### Compensação de ganho e vinheta

In [ ]:
def estimate_photometric(small_images, rotations, camera, canvas, s, samples_per_pair=200):
    keys = sorted(small_images)
    data = {}
    for k in keys:
        roi = image_roi(canvas, k, s)
        map_x, map_y, valid, r2 = warp_maps(rotations[k], camera, canvas, s, roi, s)
        img = cv2.remap(small_images[k], map_x, map_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
        lum = srgb_to_linear(img.astype(np.float32) / np.iinfo(img.dtype).max) @ np.float32([0.0722, 0.7152, 0.2126])
        data[k] = (roi, lum, cv2.erode(valid.astype(np.uint8), np.ones((5, 5), np.uint8)) > 0, r2)

    rng = np.random.default_rng(0)
    rows_i, rows_j, li, lj, ri, rj = [], [], [], [], [], []
    for a in range(len(keys)):
        for b in range(a + 1, len(keys)):
            (ra, La, Va, Ra), (rb, Lb, Vb, Rb) = data[keys[a]], data[keys[b]]
            x0, y0 = max(ra[0], rb[0]), max(ra[1], rb[1])
            x1, y1 = min(ra[2], rb[2]), min(ra[3], rb[3])
            if x1 <= x0 or y1 <= y0:
                continue
            sa = (slice(y0 - ra[1], y1 - ra[1]), slice(x0 - ra[0], x1 - ra[0]))
            sb = (slice(y0 - rb[1], y1 - rb[1]), slice(x0 - rb[0], x1 - rb[0]))
            ok = Va[sa] & Vb[sb] & (La[sa] > 0.02) & (Lb[sb] > 0.02) & (La[sa] < 0.85) & (Lb[sb] < 0.85)
            idx = np.flatnonzero(ok)
            if len(idx) < 50:
                continue
            idx = rng.choice(idx, min(samples_per_pair, len(idx)), replace=False)
            rows_i.append(np.full(len(idx), a))
            rows_j.append(np.full(len(idx), b))
            li.append(La[sa].ravel()[idx])
            lj.append(Lb[sb].ravel()[idx])
            ri.append(Ra[sa].ravel()[idx])
            rj.append(Rb[sb].ravel()[idx])

    n = len(keys)
    I, J = np.concatenate(rows_i), np.concatenate(rows_j)
    y = np.log(np.concatenate(li)) - np.log(np.concatenate(lj))
    r_i, r_j = np.concatenate(ri), np.concatenate(rj)
    m = len(y)
    A = np.zeros((m + n, n + 3))
    A[np.arange(m), I] = 1
    A[np.arange(m), J] = -1
    for p in range(3):
        A[:m, n + p] = r_i ** (p + 1) - r_j ** (p + 1)
    A[m + np.arange(n), np.arange(n)] = 0.1 * np.sqrt(m / n)
    rhs = np.concatenate([y, np.zeros(n)])
    weights = np.ones(m + n)
    for _ in range(8):
        sol, *_ = np.linalg.lstsq(A * weights[:, None], rhs * weights, rcond=None)
        res = np.abs(A[:m] @ sol - y)
        weights[:m] = np.sqrt(np.where(res <= 0.05, 1.0, 0.05 / res))
    return {k: float(np.exp(sol[a])) for a, k in enumerate(keys)}, tuple(float(c) for c in sol[n:])

### Costuras e alinhamento local

In [ ]:
def find_seams_sequential(seam_images, order, rotations, camera, canvas, s, gains, vignetting,
                          band_frac=0.25, center_weight=0.05):
    W, H = canvas_size_at(canvas, s)
    labels = np.full((H, W), -1, np.int16)
    acc = np.zeros((H, W, 3), np.float32)
    fs = canvas["f"] * s
    prev = None
    for k in order:
        x0, y0, x1, y1 = roi = image_roi(canvas, k, s)
        img, valid = warp_corrected(seam_images[k], rotations[k], camera, canvas, s, roi, s, gains[k], vignetting)
        valid = cv2.erode(valid.astype(np.uint8), np.ones((5, 5), np.uint8)) > 0
        lab = labels[y0:y1, x0:x1]
        acc_roi = acc[y0:y1, x0:x1]
        overlap = valid & (lab >= 0)
        take = valid & (lab < 0)
        if prev is not None and overlap.any():
            x_mid = (0.5 * (canvas["centers"][prev] + canvas["centers"][k]) - canvas["a_min"]) * fs - x0
            half_band = max(band_frac * (x1 - x0), 8.0)
            xx = np.arange(x1 - x0, dtype=np.float32)
            in_band = np.abs(xx - x_mid) <= half_band
            cost = cv2.blur(np.abs(acc_roi - img).mean(axis=2), (3, 3)) + center_weight * ((xx - x_mid) / half_band) ** 2
            search = overlap & in_band[None, :]
            rows = np.flatnonzero(search.any(axis=1))
            seam = np.full(y1 - y0, x_mid)
            if len(rows):
                cols = np.flatnonzero(in_band)
                r0, r1, c0, c1 = rows[0], rows[-1] + 1, cols[0], cols[-1] + 1
                local = find_optimal_seam(cost[r0:r1, c0:c1], search[r0:r1, c0:c1]) + c0
                seam[r0:r1] = local
                seam[:r0], seam[r1:] = local[0], local[-1]
            take |= overlap & (xx[None, :] > seam[:, None])
        elif prev is None:
            take = valid.copy()
        lab[take] = k
        acc_roi[take] = img[take]
        prev = k
    return labels


def flow_gray(img_a, img_b):
    ga = cv2.cvtColor(img_a, cv2.COLOR_BGR2GRAY)
    gb = cv2.cvtColor(img_b, cv2.COLOR_BGR2GRAY)
    lo, hi = np.percentile(np.concatenate([ga.ravel(), gb.ravel()]), [1, 99])
    return [(np.clip((g - lo) / max(hi - lo, 1e-6), 0, 1) ** 0.5 * 255).astype(np.uint8) for g in (ga, gb)]


def align_seams(sources, src_scale, labels, rotations, camera, canvas, s, gains, vignetting,
                fade=48, max_flow=12.0, smooth=3.0):
    W, H = canvas_size_at(canvas, s)
    lab = cv2.resize(labels, (W, H), interpolation=cv2.INTER_NEAREST).astype(np.int32)
    ids, coords = [], []
    for sl_a, sl_b in (((slice(None), slice(None, -1)), (slice(None), slice(1, None))),
                       ((slice(None, -1), slice(None)), (slice(1, None), slice(None)))):
        la, lb = lab[sl_a], lab[sl_b]
        ys, xs = np.nonzero((la != lb) & (la >= 0) & (lb >= 0))
        a, b = la[ys, xs], lb[ys, xs]
        ids.append(np.minimum(a, b) * 100000 + np.maximum(a, b))
        coords.append(np.stack([xs, ys], axis=1))
    ids, coords = np.concatenate(ids), np.concatenate(coords)

    dis = cv2.DISOpticalFlow_create(cv2.DISOPTICAL_FLOW_PRESET_MEDIUM)
    acc = {}
    for pid in np.unique(ids):
        a, b = int(pid // 100000), int(pid % 100000)
        pts = coords[ids == pid]
        if len(pts) < 20:
            continue
        m = fade + 8
        x0, y0 = max(pts[:, 0].min() - m, 0), max(pts[:, 1].min() - m, 0)
        x1, y1 = min(pts[:, 0].max() + m + 2, W), min(pts[:, 1].max() + m + 2, H)
        roi = (x0, y0, x1, y1)
        img_a, va = warp_corrected(sources[a], rotations[a], camera, canvas, s, roi, src_scale, gains[a], vignetting)
        img_b, vb = warp_corrected(sources[b], rotations[b], camera, canvas, s, roi, src_scale, gains[b], vignetting)
        both = va & vb
        if both.sum() < 100:
            continue
        ga, gb = flow_gray(img_a, img_b)
        flow_ab = dis.calc(ga, gb, None)
        flow_ba = dis.calc(gb, ga, None)
        hh, ww = ga.shape
        gx, gy = np.meshgrid(np.arange(ww, dtype=np.float32), np.arange(hh, dtype=np.float32))
        back = cv2.remap(flow_ba, gx + flow_ab[..., 0], gy + flow_ab[..., 1], cv2.INTER_LINEAR)
        fb_err = np.linalg.norm(flow_ab + back, axis=2)
        conf = (both & (fb_err < 0.5) & (np.linalg.norm(flow_ab, axis=2) < max_flow)).astype(np.float32)
        support = cv2.GaussianBlur(conf, (0, 0), smooth)
        v = cv2.GaussianBlur(flow_ab * conf[..., None], (0, 0), smooth) / np.maximum(support, 1e-6)[..., None]
        boundary = np.zeros((hh, ww), np.uint8)
        boundary[pts[:, 1] - y0, pts[:, 0] - x0] = 1
        dist = cv2.distanceTransform(1 - boundary, cv2.DIST_L2, 5)
        weight = np.clip(1 - dist / fade, 0, 1) * np.clip(support / 0.3, 0, 1)
        half = 0.5 * v * weight[..., None]
        for k, sign in ((a, -1.0), (b, 1.0)):
            acc.setdefault(k, []).append((x0, y0, sign * half))

    offsets = {}
    for k, parts in acc.items():
        ox0, oy0 = min(p[0] for p in parts), min(p[1] for p in parts)
        ox1 = max(p[0] + p[2].shape[1] for p in parts)
        oy1 = max(p[1] + p[2].shape[0] for p in parts)
        field = np.zeros((oy1 - oy0, ox1 - ox0, 2), np.float32)
        for px, py, d in parts:
            field[py - oy0:py - oy0 + d.shape[0], px - ox0:px - ox0 + d.shape[1]] += d
        offsets[k] = (ox0, oy0, np.ascontiguousarray(field[..., 0]), np.ascontiguousarray(field[..., 1]), s)
    return offsets

### Blending multibanda e recorte

In [ ]:
def gaussian_pyramid(img, levels):
    pyr = [img]
    for _ in range(levels):
        pyr.append(cv2.pyrDown(pyr[-1]))
    return pyr


def laplacian_pyramid(img, levels):
    pyr = gaussian_pyramid(img, levels)
    for l in range(levels):
        pyr[l] -= cv2.pyrUp(pyr[l + 1], dstsize=(pyr[l].shape[1], pyr[l].shape[0]))
    return pyr


def blend_labels(get_source, order, rotations, camera, canvas, s, labels, label_scale, gains, vignetting, offsets,
                 region, num_bands):
    X0, Y0, X1, Y1 = region
    W, H = X1 - X0, Y1 - Y0
    unit = 2 ** num_bands
    Wp, Hp = -(-W // unit) * unit, -(-H // unit) * unit
    r = label_scale / s
    iy = np.clip(((Y0 + np.arange(Hp) + 0.5) * r).astype(int), 0, labels.shape[0] - 1)
    ix = np.clip(((X0 + np.arange(Wp) + 0.5) * r).astype(int), 0, labels.shape[1] - 1)
    up_labels = labels[iy[:, None], ix[None, :]]
    up_labels[H:], up_labels[:, W:] = -1, -1
    margin = unit * 4
    bands = [np.zeros((Hp >> l, Wp >> l, 3), np.float32) for l in range(num_bands + 1)]
    weights = [np.zeros((Hp >> l, Wp >> l), np.float32) for l in range(num_bands + 1)]
    interpolation = cv2.INTER_LANCZOS4 if s >= 1.0 else cv2.INTER_LINEAR

    for k in order:
        ys, xs = np.nonzero(labels == k)
        if len(xs) == 0:
            continue
        x0 = max(int(xs.min() / r) - X0 - margin, 0) // unit * unit
        y0 = max(int(ys.min() / r) - Y0 - margin, 0) // unit * unit
        x1 = min(-(-(int((xs.max() + 1) / r) - X0 + 1 + margin) // unit) * unit, Wp)
        y1 = min(-(-(int((ys.max() + 1) / r) - Y0 + 1 + margin) // unit) * unit, Hp)
        if x1 <= x0 or y1 <= y0:
            continue
        src, src_scale = get_source(k)
        img, valid = warp_corrected(src, rotations[k], camera, canvas, s, (x0 + X0, y0 + Y0, x1 + X0, y1 + Y0),
                                    src_scale, gains[k], vignetting, interpolation, offsets.get(k))
        chosen = up_labels[y0:y1, x0:x1] == k
        mask = (chosen & valid).astype(np.float32) + 1e-3 * (~chosen & valid)
        lap = laplacian_pyramid(img, num_bands)
        gm = gaussian_pyramid(mask, num_bands)
        for l in range(num_bands + 1):
            sl = (slice(y0 >> l, (y0 >> l) + gm[l].shape[0]), slice(x0 >> l, (x0 >> l) + gm[l].shape[1]))
            bands[l][sl] += lap[l] * gm[l][..., None]
            weights[l][sl] += gm[l]

    result = None
    for l in range(num_bands, -1, -1):
        band = bands[l]
        bands[l] = None
        band /= np.maximum(weights[l], 1e-8)[..., None]
        if result is not None:
            band += cv2.pyrUp(result, dstsize=(band.shape[1], band.shape[0]))
        result = band
    result = result[:H, :W]
    np.clip(result, 0, 1, out=result)
    return result


def largest_valid_rectangle(mask, max_dim=1500):
    h, w = mask.shape
    f = min(1.0, max_dim / max(h, w))
    small = cv2.resize(mask.astype(np.float32), (max(int(w * f), 1), max(int(h * f), 1)),
                       interpolation=cv2.INTER_AREA) >= 0.999
    sh, sw = small.shape
    heights = np.zeros(sw, int)
    best = (0, 0, 0, 0, 0)
    for y in range(sh):
        heights = np.where(small[y], heights + 1, 0)
        stack = []
        for x in range(sw + 1):
            hx = heights[x] if x < sw else 0
            start = x
            while stack and stack[-1][1] >= hx:
                s0, s_h = stack.pop()
                if s_h * (x - s0) > best[0]:
                    best = (s_h * (x - s0), s0, y - s_h + 1, x, y + 1)
                start = s0
            stack.append((start, hx))
    _, x0, y0, x1, y1 = best
    return (int(np.ceil((x0 + 1) / f)), int(np.ceil((y0 + 1) / f)),
            int(np.floor((x1 - 1) / f)), int(np.floor((y1 - 1) / f)))

### Composição

In [ ]:
def compose_global(work_images, work_scale, load_full, rotations, camera, projection, align):
    canvas = make_canvas(rotations, camera, projection)
    order, dup = canvas["order"], canvas["dup"]
    full_size = (camera["width"], camera["height"])
    work_images, rotations = dict(work_images), dict(rotations)
    if dup is not None:
        work_images[dup], rotations[dup] = work_images[order[0]], rotations[order[0]]

    ph_scale = work_scale / 4
    gains, vignetting = estimate_photometric({k: resize_to_scale(work_images[k], full_size, ph_scale) for k in order},
                                             rotations, camera, canvas, ph_scale)
    if dup is not None:
        gains[order[0]] = gains[dup] = float(np.sqrt(gains[order[0]] * gains[dup]))
    g = np.array(list(gains.values()))

    seam_scale = work_scale / 2
    seam_images = {k: resize_to_scale(work_images[k], full_size, seam_scale) for k in order}
    labels = find_seams_sequential(seam_images, order, rotations, camera, canvas, seam_scale, gains, vignetting)
    del seam_images
    free_memory()
    offsets = align_seams(work_images, work_scale, labels, rotations, camera, canvas, work_scale, gains, vignetting,
                          **align)
    free_memory()

    def get_source(k):
        img = work_images[k] if COMPOSE_SCALE <= work_scale else load_full(order[0] if k == dup else k)
        return resize_to_scale(img, full_size, COMPOSE_SCALE), COMPOSE_SCALE

    covered = labels >= 0
    rs = COMPOSE_SCALE / seam_scale
    if dup is None:
        cx0, cy0, cx1, cy1 = largest_valid_rectangle(covered)
        x0, x1 = int(np.ceil(cx0 * rs)), int(cx1 * rs)
    else:
        fs_s = canvas["f"] * seam_scale
        cx0 = int((canvas["centers"][order[0]] - canvas["a_min"]) * fs_s)
        cols = covered[:, cx0:cx0 + int(round(2 * np.pi * fs_s))]
        cy0 = int(np.argmax(cols, axis=0).max())
        cy1 = int(covered.shape[0] - np.argmax(cols[::-1], axis=0).max())
        turn = int(round(2 * np.pi * canvas["f"] * COMPOSE_SCALE))
        margin = turn // 32
        x0 = int(round(cx0 * rs)) - margin
        x1 = x0 + turn + 2 * margin
    y0, y1 = int(np.ceil(cy0 * rs)), int(cy1 * rs)

    bands = max(1, int(round(NUM_BANDS + np.log2(COMPOSE_SCALE))))
    panorama = blend_labels(get_source, order, rotations, camera, canvas, COMPOSE_SCALE, labels, seam_scale, gains,
                            vignetting, offsets, (x0, y0, x1, y1), bands)
    free_memory()
    if dup is not None:
        gray = panorama.mean(axis=2)
        k = min(64, margin)
        shifts = np.arange(-margin + k, margin)
        diffs = [np.abs(gray[:, margin + t] - gray[:, margin + t + turn]).mean() for t in shifts]
        first_col = margin + int(shifts[int(np.argmin(diffs))])
        cut = panorama[:, first_col:first_col + turn]
        alpha = np.linspace(0, 1, k, dtype=np.float32)[None, :, None]
        cut[:, -k:] = (1 - alpha) * cut[:, -k:] + alpha * panorama[:, first_col - k:first_col]
        panorama = cut

    span = 360.0 if dup is not None else float(np.degrees(canvas["a_max"] - canvas["a_min"]))
    return panorama, {"largura_angular_graus": span, "volta_completa_360": dup is not None,
                      "ganho_min": float(g.min()), "ganho_max": float(g.max()),
                      "vinheta_cantos_EV": float(np.log2(vignetting_factor(1.0, vignetting)))}

### Pipeline dos extras

In [ ]:
def load_scaled(input_dir, max_dim):
    filenames = sorted(f for f in os.listdir(input_dir) if f.endswith(".png"))
    random.Random(SEED).shuffle(filenames)

    def read(fname):
        img = cv2.imread(str(input_dir / fname), cv2.IMREAD_UNCHANGED)[..., :3]
        full_size = (img.shape[1], img.shape[0])
        scale = min(1.0, max_dim / max(full_size))
        if scale < 1.0:
            size = (int(round(img.shape[1] * scale)), int(round(img.shape[0] * scale)))
            img = cv2.resize(img, size, interpolation=cv2.INTER_AREA)
        return img, full_size

    with ThreadPoolExecutor(8) as ex:
        images, sizes = zip(*ex.map(read, filenames))
    return list(images), filenames, list(sizes)


def to_jpg(img):
    out8 = np.empty(img.shape, np.uint8)
    for r in range(0, img.shape[0], 256):
        out8[r:r + 256] = np.round(np.clip(img[r:r + 256], 0, 1) * 255)
    return out8


def run_extra(name, projection, settings):
    t_start = time.time()
    input_dir = ROOT / f"{name}_png"
    images, filenames, sizes = load_scaled(input_dir, WORK_DIM)

    all_kps, all_descs = zip(*(detect_features(img, DETECTOR, equalize=True) for img in images))
    connectivity, homographies, inlier_points = build_connectivity_matrix(all_kps, all_descs, RANSAC_THRESH, BETA)
    free_memory()
    graph = build_neighbor_graph(connectivity, MIN_INLIERS)
    intruders = detect_intruders(graph)
    valid_nodes = [i for i in range(len(images)) if i not in intruders]
    order, closed = infer_order_spectral(graph, valid_nodes)
    pairs = list(zip(order, order[1:]))
    print(f"{name}: {len(images)} fotos, intrusas {[filenames[i] for i in intruders]}, fecha 360 graus: {closed}")

    full_w, full_h = sizes[valid_nodes[0]]
    work_scale = images[valid_nodes[0]].shape[1] / full_w
    neighbors = np.where(connectivity >= MIN_INLIERS, connectivity, 0)
    camera = make_camera(full_w, full_h, 1.0)
    focal = estimate_focal({p: H for p, H in homographies.items() if neighbors[p] > 0}, work_scale, camera)
    camera = make_camera(full_w, full_h, focal or 0.9 * max(full_w, full_h))
    ref = order[len(order) // 2]
    rotations = initial_rotations(homographies, neighbors, valid_nodes, ref, work_scale, camera)
    ba_pairs = {p: pts for p, pts in inlier_points.items()
                if neighbors[p] > 0 and p[0] in rotations and p[1] in rotations}
    rotations, camera, ba_errors = bundle_adjust(ba_pairs, rotations, camera, work_scale, ref, **settings["ba"])
    rotations = wave_correct(rotations)
    consecutive = [round(ba_errors[p], 3) for p in (tuple(sorted(p)) for p in pairs) if p in ba_errors]
    del all_kps, all_descs, homographies, inlier_points, ba_pairs
    free_memory()

    def load_full(k):
        return cv2.imread(str(input_dir / filenames[k]), cv2.IMREAD_UNCHANGED)[..., :3]

    panorama, compose_metrics = compose_global({k: images[k] for k in rotations}, work_scale, load_full, rotations,
                                               camera, projection, settings["align"])
    panorama = to_jpg(panorama)
    cv2.imwrite(out(EXTRAS_RESULTS, name, "6_composicao", "panorama_final.jpg"), panorama)

    metrics = {"fotos_no_panorama": len(rotations), "intrusas_rejeitadas": len(intruders),
               "largura_px": panorama.shape[1], "altura_px": panorama.shape[0], **compose_metrics,
               "erro_BA_mediana_pares_consecutivos_px": float(np.median(consecutive)),
               "focal_px": camera["f"], "tempo_total_s": round(time.time() - t_start)}
    metrics = {k: round(v, 3) if isinstance(v, float) else v for k, v in metrics.items()}
    write_csv(out(EXTRAS_RESULTS, name, "6_composicao", "metricas.csv"),
              [{"metrica": k, "valor": v} for k, v in metrics.items()])
    f = 2400 / panorama.shape[1]
    preview = cv2.resize(panorama, None, fx=f, fy=f, interpolation=cv2.INTER_AREA)
    return {"metricas": metrics, "pares": [(filenames[i], filenames[j]) for i, j in pairs], "previa": preview}

### Extra 1: D1 com bundle adjustment e projeção cilíndrica

In [ ]:
extras = {}
extras["D1"] = run_extra("D1", *EXTRAS["D1"])
show(extras["D1"]["previa"])

### Extra 1: A1, 61 fotos em 213°

In [ ]:
extras["A1"] = run_extra("A1", *EXTRAS["A1"])
show(extras["A1"]["previa"])

### Extra 2: B1, panorama 360° em projeção esférica

In [ ]:
extras["B1"] = run_extra("B1", *EXTRAS["B1"])
show(extras["B1"]["previa"])

### Extra 3: compensação de exposição

Ganhos por foto e vinheta da lente estimados nas sobreposições; resumo dos três panoramas.

In [ ]:
table([{"conjunto": name, **result["metricas"]} for name, result in extras.items()])

### Extra 4: SuperPoint + LightGlue x SIFT

Pares consecutivos do A1, na escala de 3000 px; RANSAC com limiar de 4 px e erro em pixels da resolução total.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sync = torch.cuda.synchronize if device.type == "cuda" else (lambda: None)
extractor = SuperPoint(max_num_keypoints=8192).eval().to(device)
matcher = LightGlue(features="superpoint").eval().to(device)


def load_x4(path):
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)[..., :3]
    s = min(1.0, WORK_DIM / max(img.shape[:2]))
    return cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA), s


def match_sift(img_a, img_b):
    t0 = time.perf_counter()
    kps_a, descs_a = detect_features(img_a, "SIFT", equalize=True)
    kps_b, descs_b = detect_features(img_b, "SIFT", equalize=True)
    good = match_features(descs_a, descs_b)
    elapsed = time.perf_counter() - t0
    pts_a = np.float32([kps_a[m.queryIdx].pt for m in good])
    pts_b = np.float32([kps_b[m.trainIdx].pt for m in good])
    return (len(kps_a) + len(kps_b)) / 2, pts_a, pts_b, elapsed


def match_lightglue(img_a, img_b):
    def to_tensor(img):
        return torch.from_numpy(to_gray8(img, equalize=True)).float()[None].to(device) / 255

    sync()
    t0 = time.perf_counter()
    with torch.inference_mode():
        feats_a = extractor.extract(to_tensor(img_a), resize=None)
        feats_b = extractor.extract(to_tensor(img_b), resize=None)
        matches = matcher({"image0": feats_a, "image1": feats_b})
    feats_a, feats_b, matches = rbd(feats_a), rbd(feats_b), rbd(matches)
    idx = matches["matches"].cpu().numpy()
    pts_a = feats_a["keypoints"].cpu().numpy()[idx[:, 0]].astype(np.float32)
    pts_b = feats_b["keypoints"].cpu().numpy()[idx[:, 1]].astype(np.float32)
    sync()
    return (len(feats_a["keypoints"]) + len(feats_b["keypoints"])) / 2, pts_a, pts_b, time.perf_counter() - t0


def ransac_metrics(pts_a, pts_b, scale):
    if len(pts_a) < 4:
        return 0, 0.0, float("nan")
    H, mask = cv2.findHomography(pts_a, pts_b, cv2.RANSAC, RANSAC_THRESH)
    if H is None:
        return 0, 0.0, float("nan")
    mask = mask.ravel().astype(bool)
    proj = cv2.perspectiveTransform(pts_a[mask].reshape(-1, 1, 2), H).reshape(-1, 2)
    return int(mask.sum()), float(mask.mean()), float(np.linalg.norm(proj - pts_b[mask], axis=1).mean()) / scale

In [ ]:
pairs = extras[X4_DATASET]["pares"]
x4_images = {f: load_x4(ROOT / f"{X4_DATASET}_png" / f) for f in dict.fromkeys(f for pair in pairs for f in pair)}
match_lightglue(x4_images[pairs[0][0]][0], x4_images[pairs[0][1]][0])

methods = {"SIFT + ratio test (CPU)": match_sift, "SuperPoint + LightGlue (GPU)": match_lightglue}
x4_rows = []
for k, (fa, fb) in enumerate(pairs):
    (img_a, scale), (img_b, _) = x4_images[fa], x4_images[fb]
    for method, run in methods.items():
        n_kps, pts_a, pts_b, elapsed = run(img_a, img_b)
        inliers, rate, error = ransac_metrics(pts_a, pts_b, scale)
        x4_rows.append({"par": k, "metodo": method, "keypoints": round(n_kps), "matches": len(pts_a),
                        "inliers": inliers, "taxa_inliers": round(rate, 4), "erro_reproj_px": round(error, 3),
                        "tempo_s": round(elapsed, 3)})

x4_summary = [{"metodo": method, **{key: round(float(np.nanmean([r[key] for r in x4_rows if r["metodo"] == method])), 3)
                                    for key in ("keypoints", "matches", "inliers", "taxa_inliers", "erro_reproj_px",
                                                "tempo_s")}}
              for method in methods]
write_csv(out(EXTRAS_RESULTS, "x4", "sift_vs_lightglue.csv"), x4_summary)
print(f"{len(pairs)} pares consecutivos do {X4_DATASET}")
table(x4_summary)
del x4_images